# Ćwiczenie 4.5: szybka analiza automatyczna — od drzewa do XGBoost

Ten notebook jest krótkim podsumowaniem zajęć o metodach drzewiastych.

W poprzednich notebookach rozbijaliśmy algorytmy na części:

```text
pojedyncze drzewo -> Random Forest -> Gradient Boosting -> XGBoost
```

Tutaj przechodzimy do praktycznego workflow:

```text
dane -> preprocessing -> kilka modeli -> porównanie metryk -> wybór progu -> ranking obserwacji
```

Nie liczymy już ręcznie każdego Gini, residualu i gainu.  
Celem jest zobaczenie, jak zdobyta wiedza przekłada się na szybką analizę danych tabelarycznych.

W notebooku porównujemy baseline, pojedyncze drzewo, Random Forest, Gradient Boosting i XGBoost.
Wyniki nie są zakładane z góry — modele są porównywane na tych samych danych i tych samych metrykach.


## 0. Dlaczego taki zbiór danych?

Używamy syntetycznego, ale biznesowo realistycznego zbioru **churn klientów**.

Dlaczego taki przykład jest wygodny dydaktycznie?

- `tips` jest świetny do podstaw, ale jest za mały i zbyt prosty do porównywania silniejszych modeli.
- Titanic dobrze pokazuje preprocessing, ale często dużo czasu zajmuje samo czyszczenie danych.
- Dane maratończyków mogą być ciekawe, ale zwykle wymagają dłuższego przygotowania celu analizy.
- Churn dobrze pasuje do tego rozdziału, bo mamy klasyfikację, próg decyzyjny, ranking ryzyka i koszt pomyłek.

Dane generujemy lokalnie, więc notebook działa bez internetu.  
W danych są celowo zaszyte zależności nieliniowe, np.:

```text
dużo reklamacji + umowa miesięczna -> większe ryzyko churn
mało logowań + oferta konkurencji -> większe ryzyko churn
długi staż + rabat -> mniejsze ryzyko churn
```

Dodajemy też pola opisowe:

```text
customer_id
customer_name
```

Służą one do raportowania i rankingu na końcu notebooka. Nie powinny być używane jako cechy modelu.
Model ma uczyć się wzorców zachowania klientów, a nie zapamiętywać identyfikatory lub nazwy.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception as exc:
    HAS_XGBOOST = False
    XGBClassifier = None
    print("XGBoost nie jest dostępny w tym środowisku. Notebook użyje modeli sklearn.")

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

RANDOM_STATE = 9


## 1. Dane: syntetyczny churn

Każdy wiersz to klient.

Wybrane cechy predykcyjne:

- `tenure_months` — staż klienta,
- `complaints_count` — liczba reklamacji,
- `avg_logins_week` — średnia liczba logowań tygodniowo,
- `late_payments` — spóźnione płatności,
- `contract_type` — typ umowy,
- `competitor_offer` — czy klient dostał ofertę konkurencji,
- `has_discount` — czy ma rabat,
- `churn` — zmienna docelowa: 1 oznacza odejście klienta.

Pola opisowe:

- `customer_id`,
- `customer_name`.

Te pola zostawiamy na końcu tabeli. Przydadzą się w rankingu klientów, ale zostaną wykluczone z treningu modelu.


In [ ]:
def make_synthetic_churn(n=3000, random_state=42):
    rng = np.random.default_rng(random_state)

    tenure = rng.uniform(0, 72, n).round(1)
    complaints = rng.poisson(0.8 + 1.1 * (tenure < 12), n)
    complaints = np.clip(complaints, 0, 7)
    logins = np.clip(
        rng.normal(5.5 - 0.4 * complaints + 0.03 * tenure, 1.4, n),
        0,
        12,
    ).round(1)
    late_payments = rng.binomial(
        4,
        np.clip(0.06 + 0.05 * complaints + 0.03 * (tenure < 6), 0.01, 0.5),
        n,
    )

    contract_type = rng.choice(["miesieczna", "roczna", "dwuletnia"], n, p=[0.50, 0.32, 0.18])
    plan_type = rng.choice(["basic", "standard", "premium"], n, p=[0.42, 0.41, 0.17])
    region = rng.choice(["A", "B", "C", "D"], n, p=[0.35, 0.25, 0.25, 0.15])
    competitor_offer = rng.binomial(
        1,
        np.clip(0.12 + 0.10 * (contract_type == "miesieczna") + 0.05 * complaints, 0.02, 0.75),
        n,
    )
    usage_gb = np.clip(
        rng.normal(70 + 20 * (plan_type == "premium") - 10 * (plan_type == "basic"), 25, n),
        1,
        200,
    ).round(1)
    has_discount = rng.binomial(
        1,
        np.clip(0.15 + 0.005 * tenure + 0.10 * (contract_type != "miesieczna"), 0.03, 0.70),
        n,
    )

    # Ukryta funkcja ryzyka. Modele jej nie widzą.
    # Modele dostają tylko cechy wejściowe i etykietę churn.
    score = -2.2
    score += 0.45 * (tenure < 6) + 0.35 * (tenure < 12) + 0.20 * (tenure < 24)
    score += 0.45 * (complaints >= 1) + 0.55 * (complaints >= 2) + 0.45 * (complaints >= 3)
    score += 0.60 * (logins < 2.5) + 0.25 * (logins < 4)
    score += 0.45 * (late_payments >= 1) + 0.55 * (late_payments >= 2)
    score += 0.55 * (contract_type == "miesieczna") - 0.35 * (contract_type == "dwuletnia")
    score += 0.55 * competitor_offer + 0.30 * (plan_type == "basic") - 0.35 * has_discount

    # Interakcje: tu metody drzewiaste mają przewagę nad prostą pojedynczą regułą.
    score += 0.50 * ((contract_type == "miesieczna") & (complaints >= 2))
    score += 0.60 * ((competitor_offer == 1) & (logins < 3))
    score += -0.50 * ((has_discount == 1) & (tenure > 24))

    # Szum: problem nie jest idealnie deterministyczny.
    score += rng.normal(0, 0.5, n)

    probability = 1 / (1 + np.exp(-score))
    churn = rng.binomial(1, probability)

    # Nazwy są sztuczne i służą tylko do raportowania.
    customer_id = np.arange(1, n + 1)
    customer_name = [f"Klient_{i:04d}" for i in customer_id]

    df = pd.DataFrame({
        "tenure_months": tenure,
        "complaints_count": complaints,
        "avg_logins_week": logins,
        "late_payments": late_payments,
        "contract_type": contract_type,
        "plan_type": plan_type,
        "region": region,
        "competitor_offer": competitor_offer,
        "usage_gb": usage_gb,
        "has_discount": has_discount,
        "churn": churn,
        "customer_id": customer_id,
        "customer_name": customer_name,
    })

    # Dodajemy kilka braków danych, żeby pipeline z imputerem miał sens.
    for col in ["avg_logins_week", "usage_gb"]:
        missing_mask = rng.random(n) < 0.03
        df.loc[missing_mask, col] = np.nan

    # Celowo zostawiamy identyfikator i nazwę na końcu.
    # Są potrzebne do raportów, ale nie będą używane przez model.
    ordered_columns = [
        "tenure_months",
        "complaints_count",
        "avg_logins_week",
        "late_payments",
        "contract_type",
        "plan_type",
        "region",
        "competitor_offer",
        "usage_gb",
        "has_discount",
        "churn",
        "customer_id",
        "customer_name",
    ]
    return df[ordered_columns]


df = make_synthetic_churn(n=2500, random_state=RANDOM_STATE)
display(df.head())
print("Liczba obserwacji:", len(df))
print("Odsetek churn:", round(df["churn"].mean(), 3))



## 2. Szybki EDA: czy dane mają sygnał?

Nie robimy pełnej analizy eksploracyjnej.  
Wystarczą 2–3 szybkie kontrole:

- jaki jest odsetek churn,
- czy reklamacje wiążą się z większym churn,
- czy typ umowy ma znaczenie.


In [ ]:

print("Rozkład targetu:")
display(df["churn"].value_counts(normalize=True).rename("proportion").to_frame())

print("Churn według liczby reklamacji:")
display(
    df.assign(complaints_bucket=pd.cut(df["complaints_count"], bins=[-1, 0, 1, 2, 7], labels=["0", "1", "2", "3+"]))
      .groupby("complaints_bucket", observed=True)["churn"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "churn_rate"})
      .round(3)
)

print("Churn według typu umowy:")
display(
    df.groupby("contract_type")["churn"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "churn_rate"})
      .sort_values("churn_rate", ascending=False)
      .round(3)
)



## 3. Podział danych i pipeline preprocessingu

Robimy trzy zbiory:

```text
train      -> uczymy modele
validation -> wybieramy próg decyzyjny
test       -> końcowa ocena
```

To ważne: progu decyzyjnego nie dobieramy na teście, bo wtedy wynik testowy byłby zbyt optymistyczny.


In [ ]:
TARGET = "churn"
ID_COL = "customer_id"
NAME_COLS = ["customer_name"]
DROP_FROM_MODEL = [ID_COL] + NAME_COLS

# X zawiera też pola opisowe, bo chcemy je zachować do raportów.
# Funkcja make_preprocessor wykluczy je z treningu modelu.
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=RANDOM_STATE
)

print("train:", X_train.shape, "valid:", X_valid.shape, "test:", X_test.shape)
print("Kolumny wykluczone z modelowania:", DROP_FROM_MODEL)


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocessor(X_frame):
    features = X_frame.drop(columns=DROP_FROM_MODEL, errors="ignore")
    numeric_features = features.select_dtypes(include="number").columns.tolist()
    categorical_features = features.select_dtypes(exclude="number").columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ])

    return ColumnTransformer([
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ])



## 4. Porównujemy modele automatycznie

Modele:

- `DummyClassifier` — baseline, czyli punkt odniesienia,
- `DecisionTreeClassifier` — pojedyncze drzewo,
- `RandomForestClassifier` — wiele niezależnych drzew,
- `GradientBoostingClassifier` — drzewa jako kolejne poprawki,
- `XGBoost` — boosting drzew z regularizacją i bardzo dopracowaną implementacją.

Nie zakładamy z góry, że XGBoost zawsze wygra.  
W praktyce uczciwie porównujemy modele i patrzymy na metryki.


In [ ]:

models = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        min_samples_leaf=10,
        max_features="sqrt",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=160,
        learning_rate=0.05,
        max_depth=2,
        random_state=RANDOM_STATE,
    ),
}

if HAS_XGBOOST:
    models["XGBoost"] = XGBClassifier(
        n_estimators=180,
        learning_rate=0.10,
        max_depth=2,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=2,
        min_child_weight=3,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=2,
    )


def threshold_report(y_true, proba, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.05, 0.91, 0.01), 2)

    rows = []
    for thr in thresholds:
        pred = (proba >= thr).astype(int)
        rows.append({
            "threshold": thr,
            "precision_tak": precision_score(y_true, pred, zero_division=0),
            "recall_tak": recall_score(y_true, pred, zero_division=0),
            "f1_tak": f1_score(y_true, pred, zero_division=0),
            "predicted_tak": int(pred.sum()),
        })
    return pd.DataFrame(rows)


def get_best_threshold_on_validation(y_valid, proba_valid):
    report = threshold_report(y_valid, proba_valid)
    # Wybieramy próg maksymalizujący F1 dla klasy churn=1.
    # Przy remisie bierzemy większy recall, bo w churn często wolimy złapać więcej klientów ryzyka.
    report_sorted = report.sort_values(["f1_tak", "recall_tak"], ascending=False)
    return float(report_sorted.iloc[0]["threshold"]), report


fitted_models = {}
reports_by_model = {}
results = []

for name, model in models.items():
    pipe = Pipeline([
        ("preprocess", make_preprocessor(X_train)),
        ("model", model),
    ])
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe

    proba_valid = pipe.predict_proba(X_valid)[:, 1]
    best_thr, valid_thresholds = get_best_threshold_on_validation(y_valid, proba_valid)
    reports_by_model[name] = valid_thresholds

    proba_test = pipe.predict_proba(X_test)[:, 1]
    pred_05 = (proba_test >= 0.50).astype(int)
    pred_best = (proba_test >= best_thr).astype(int)

    results.append({
        "model": name,
        "accuracy@0.50": accuracy_score(y_test, pred_05),
        "f1_tak@0.50": f1_score(y_test, pred_05, zero_division=0),
        "roc_auc": roc_auc_score(y_test, proba_test),
        "log_loss": log_loss(y_test, proba_test),
        "prog_z_valid": best_thr,
        "precision_tak@test": precision_score(y_test, pred_best, zero_division=0),
        "recall_tak@test": recall_score(y_test, pred_best, zero_division=0),
        "f1_tak@test": f1_score(y_test, pred_best, zero_division=0),
    })

results_df = pd.DataFrame(results).sort_values("f1_tak@test", ascending=False)
display(results_df.round(3))



## 5. Co widzimy?

Patrzymy na dwie rzeczy:

1. Czy modele drzewiaste biją baseline?
2. Czy modele zespołowe, szczególnie boosting/XGBoost, biją pojedyncze drzewo?

W praktyce nie interesuje nas tylko `accuracy`.  
Przy churn ważne są też:

- `recall_tak` — ilu odchodzących klientów wykrywamy,
- `precision_tak` — jak dużo alarmów jest trafnych,
- `f1_tak` — kompromis precision/recall,
- `roc_auc` — jakość rankingu ryzyka.


In [ ]:

best_model_name = results_df.iloc[0]["model"]
best_threshold = float(results_df.iloc[0]["prog_z_valid"])
best_model = fitted_models[best_model_name]

print("Najlepszy model według f1_tak@test:", best_model_name)
print("Próg wybrany na validation:", best_threshold)

fig, ax = plt.subplots(figsize=(8, 4))
plot_df = results_df.sort_values("f1_tak@test", ascending=True)
ax.barh(plot_df["model"], plot_df["f1_tak@test"])
ax.set_xlabel("F1 dla klasy churn=1 na teście")
ax.set_title("Porównanie modeli po dobraniu progu na validation")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
plot_df = results_df.sort_values("roc_auc", ascending=True)
ax.barh(plot_df["model"], plot_df["roc_auc"])
ax.set_xlabel("ROC AUC na teście")
ax.set_title("Jakość rankingu ryzyka")
plt.show()


In [ ]:

# TODO B1: interpretacja wyniku.
#
# 1. Który model wygrał według f1_tak@test?
# 2. Czy pojedyncze drzewo było lepsze czy gorsze od metod zespołowych?
# 3. Czy XGBoost wygrał, czy był blisko najlepszych modeli?
#
# Wpisz krótką odpowiedź tekstową.

interpretacja_modeli_student = ...

if interpretacja_modeli_student is Ellipsis:
    print("Uzupełnij krótką interpretację wyników.")
else:
    print(interpretacja_modeli_student)



## 6. Próg decyzyjny: model daje prawdopodobieństwo, a decyzję podejmujemy my

Model zwraca prawdopodobieństwo churn, np.:

```text
klient A: 0.18
klient B: 0.63
klient C: 0.41
```

Ale biznes musi zdecydować, od jakiego progu klient trafia do akcji retencyjnej.

Przykład:

- próg `0.50` — ostrożniejszy, mniej kontaktów,
- próg `0.30` — więcej wykrytych klientów ryzyka, ale więcej fałszywych alarmów.

To jest dokładnie to samo myślenie, które pojawiło się wcześniej przy ćwiczeniu F w Random Forest.


In [ ]:

proba_valid_best = best_model.predict_proba(X_valid)[:, 1]
threshold_df = threshold_report(y_valid, proba_valid_best)

display(
    threshold_df.loc[threshold_df["threshold"].isin([0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50])]
    .round(3)
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_df["threshold"], threshold_df["precision_tak"], label="precision_tak")
ax.plot(threshold_df["threshold"], threshold_df["recall_tak"], label="recall_tak")
ax.plot(threshold_df["threshold"], threshold_df["f1_tak"], label="f1_tak")
ax.axvline(best_threshold, linestyle="--", label=f"wybrany próg = {best_threshold:.2f}")
ax.set_xlabel("Próg decyzyjny")
ax.set_ylabel("Wartość metryki")
ax.set_title(f"Precision / recall / F1 dla modelu: {best_model_name}")
ax.legend()
plt.show()


In [ ]:

# TODO B2: wybór progu biznesowego.
#
# Załóżmy, że dział retencji mówi:
#     chcemy recall_tak co najmniej 0.65,
#     a spośród takich progów wybierzmy najwyższe precision_tak.
#
# Uzupełnij business_threshold_student na podstawie tabeli threshold_df.

candidate_thresholds = threshold_df[threshold_df["recall_tak"] >= 0.65].copy()
if len(candidate_thresholds) > 0:
    suggested_business_threshold = float(
        candidate_thresholds.sort_values("precision_tak", ascending=False).iloc[0]["threshold"]
    )
else:
    suggested_business_threshold = best_threshold

business_threshold_student = ...

print("Sugerowany próg według reguły biznesowej:", suggested_business_threshold)

if business_threshold_student is Ellipsis:
    print("Uzupełnij business_threshold_student.")
else:
    print("Wybrany próg:", business_threshold_student)


## 7. Co model uznał za ważne?

W modelach drzewiastych `feature_importances_` jest miarą **ważności predykcyjnej cech w konkretnym dopasowanym modelu**.

Najprościej:

```text
cecha ma większą ważność,
jeżeli splity używające tej cechy często i mocno poprawiały jakość drzewa.
```

W klasycznych drzewach i Random Forest w scikit-learn jest to zwykle znormalizowana suma spadków nieczystości
(np. Gini) przypisana do splitów wykorzystujących daną cechę. W modelach boostingowych i XGBoost definicja może
zależeć od implementacji, ale sens praktyczny jest podobny: cecha często pomagała modelowi poprawiać predykcje.

Ważne rozróżnienie:

- to jest informacja **modelowa/predykcyjna**,
- nie jest to automatyczny dowód, że cecha **powoduje** churn.

Przykład: `complaints_count` może być bardzo ważna dla predykcji churn, bo dobrze rozdziela klientów ryzyka.
To nadal nie rozstrzyga samo w sobie, czy reklamacje są przyczyną odejścia, objawem szerszego problemu,
czy tylko mocnym sygnałem ostrzegawczym.


In [ ]:

def get_feature_names_from_pipeline(pipe):
    preprocessor = pipe.named_steps["preprocess"]
    feature_names = []

    for name, transformer, columns in preprocessor.transformers_:
        if name == "remainder" and transformer == "drop":
            continue
        if name == "num":
            feature_names.extend(list(columns))
        elif name == "cat":
            onehot = transformer.named_steps["onehot"]
            feature_names.extend(onehot.get_feature_names_out(columns).tolist())
    return feature_names


def get_feature_importance(pipe):
    model = pipe.named_steps["model"]
    feature_names = get_feature_names_from_pipeline(pipe)

    if hasattr(model, "feature_importances_"):
        return (
            pd.DataFrame({
                "feature": feature_names,
                "importance": model.feature_importances_,
            })
            .sort_values("importance", ascending=False)
            .reset_index(drop=True)
        )
    return None


importance_df = get_feature_importance(best_model)

if importance_df is None:
    print("Ten model nie ma feature_importances_.")
else:
    display(importance_df.head(15).round(4))

    top = importance_df.head(12).sort_values("importance", ascending=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(top["feature"], top["importance"])
    ax.set_xlabel("Ważność cechy")
    ax.set_title(f"Najważniejsze cechy według modelu: {best_model_name}")
    plt.show()



## 8. Ranking klientów: po co nam prawdopodobieństwa?

W churn często nie chodzi tylko o klasę `tak/nie`.

Bardziej praktyczne pytanie brzmi:

```text
których 20 klientów ma największe ryzyko i warto ich obsłużyć w pierwszej kolejności?
```

Dlatego zaawansowane modele są przydatne nie tylko jako klasyfikatory, ale też jako narzędzia do **rankingu ryzyka**.


In [ ]:
proba_test_best = best_model.predict_proba(X_test)[:, 1]
pred_business = (proba_test_best >= business_threshold_student if business_threshold_student is not Ellipsis else proba_test_best >= best_threshold).astype(int)

scored_test = X_test.copy()
scored_test["true_churn"] = y_test.values
scored_test["predicted_churn_probability"] = proba_test_best
scored_test["predicted_action"] = pred_business

# W raporcie pokazujemy najpierw wynik modelu i cechy wyjaśniające,
# a identyfikator oraz nazwę zostawiamy na końcu.
top_risk_columns = [
    "predicted_churn_probability",
    "predicted_action",
    "true_churn",
    "contract_type",
    "complaints_count",
    "avg_logins_week",
    "late_payments",
    "competitor_offer",
    "tenure_months",
    "has_discount",
    "customer_id",
    "customer_name",
]

top_risk = scored_test.sort_values("predicted_churn_probability", ascending=False).head(15)
display(top_risk[top_risk_columns].round(3))

cm = confusion_matrix(y_test, pred_business)
disp = ConfusionMatrixDisplay(cm, display_labels=["nie", "tak"])
disp.plot(values_format="d")
plt.title(f"Macierz pomyłek: {best_model_name}, próg={business_threshold_student if business_threshold_student is not Ellipsis else best_threshold:.2f}")
plt.show()


In [ ]:

# TODO B3: interpretacja rankingu.
#
# Popatrz na top_risk.
# Czy klienci o najwyższym ryzyku mają cechy, które pasują do intuicji z wcześniejszych notebooków?
# Np. dużo reklamacji, krótki staż, mało logowań, oferta konkurencji, umowa miesięczna.

interpretacja_rankingu_student = ...

if interpretacja_rankingu_student is Ellipsis:
    print("Uzupełnij krótką interpretację rankingu klientów.")
else:
    print(interpretacja_rankingu_student)


## 9. Co z tego wynika?

Ten notebook nie zastępuje poprzednich ćwiczeń. Pokazuje, po co były potrzebne.

```text
4.1  Gini i pojedyncze drzewo
4.2  progi liczbowe i regresja
4.3  Random Forest: wiele niezależnych drzew
4.4  Gradient Boosting / XGBoost: suma kolejnych poprawek
4.5  szybki workflow: dane -> modele -> metryki -> próg -> ranking
```

Najważniejsze wnioski:

1. Baseline jest obowiązkowy, bo bez niego trudno ocenić, czy model daje realną wartość.
2. Pojedyncze drzewo jest dobre do zrozumienia mechaniki, ale zwykle bywa mniej stabilne predykcyjnie.
3. Random Forest stabilizuje drzewa przez uśrednianie wielu modeli.
4. Gradient Boosting i XGBoost uczą się sekwencyjnie, poprawiając błędy poprzedniego modelu.
5. XGBoost jest często mocnym modelem startowym dla danych tabelarycznych, ale nadal trzeba go porównać z alternatywami.
6. W praktyce równie ważny jak model jest próg decyzyjny i sposób użycia prawdopodobieństw.
7. Pola typu ID i nazwa przydają się do raportowania, ale nie powinny być cechami treningowymi.


## Appendix A: jak szybko podmienić dane?

Ten sam schemat można zastosować do innych zbiorów danych.

Minimalna konfiguracja na początku analizy:

```python
TARGET = "nazwa_kolumny_docelowej"
ID_COL = "id"              # opcjonalnie
NAME_COLS = ["nazwa"]      # opcjonalnie
DROP_FROM_MODEL = [ID_COL] + NAME_COLS
```

Zasada:

```text
kolumny opisowe, takie jak ID albo nazwa, zostają w tabeli do raportu,
ale są wykluczone z uczenia modelu.
```

Przykładowe konfiguracje:

| Zbiór danych | Typ zadania | TARGET | ID / nazwa | Uwaga |
|---|---|---|---|---|
| Churn klientów | klasyfikacja | `churn` | `customer_id`, `customer_name` | dobry do progu decyzyjnego i rankingu ryzyka |
| Titanic | klasyfikacja | `survived` | `passenger_id`, `name` | dobry do preprocessingu, ale wymaga czyszczenia |
| Marketing response | klasyfikacja | `response` | `client_id`, `client_name` | podobny workflow jak churn |
| Breast Cancer Wisconsin | klasyfikacja | `target` | brak | wygodny offline, ale interpretację medyczną trzeba prowadzić ostrożnie |
| Wine Quality | klasyfikacja/regresja | `quality` | brak | można zrobić klasyfikację jakości albo regresję |
| California Housing | regresja | `MedHouseVal` | brak | wymaga zamiany metryk klasyfikacji na metryki regresji |

W przypadku regresji zmienia się głównie model i metryki, np. zamiast `precision`, `recall`, `f1` używa się `MAE`, `RMSE`, `R2`.


## Appendix B: przypomnienie metryk i statystyk używanych w notebooku

### `accuracy`

Odsetek wszystkich poprawnych klasyfikacji.

```text
accuracy = (TP + TN) / (TP + FP + FN + TN)
```

Dobra jako szybki punkt odniesienia, ale przy nierównych klasach może być myląca.

### `precision_tak`

Spośród klientów oznaczonych jako `churn=1`, jaki odsetek naprawdę odszedł.

```text
precision_tak = TP / (TP + FP)
```

Wysoka precision oznacza mało fałszywych alarmów.

### `recall_tak`

Spośród wszystkich klientów, którzy naprawdę odeszli, jaki odsetek model wykrył.

```text
recall_tak = TP / (TP + FN)
```

Wysoki recall oznacza mało przeoczonych klientów ryzyka.

### `f1_tak`

Średnia harmoniczna precision i recall dla klasy `tak`.

```text
f1 = 2 * precision * recall / (precision + recall)
```

Przydatna, gdy zależy nam na kompromisie między liczbą trafnych alarmów i liczbą wykrytych przypadków.

### `roc_auc`

Miara jakości rankingu prawdopodobieństw.  
Wysokie `roc_auc` oznacza, że model zwykle nadaje wyższe ryzyko obserwacjom pozytywnym niż negatywnym.

### `log_loss`

Miara jakości prawdopodobieństw. Karze pewne, ale błędne predykcje.

```text
niższy log_loss = lepiej skalibrowane / trafniejsze prawdopodobieństwa
```

### `threshold`

Próg decyzyjny zamienia prawdopodobieństwo na klasę.

```text
jeśli p(churn) >= threshold -> przewidujemy churn=1
jeśli p(churn) < threshold  -> przewidujemy churn=0
```

Niższy próg zwykle zwiększa `recall_tak`, ale może obniżać `precision_tak`.

### `predicted_tak`

Liczba obserwacji, dla których model przewidział klasę `tak` przy danym progu.

### Macierz pomyłek

```text
TP: klient miał churn i model przewidział churn
FP: klient nie miał churn, ale model przewidział churn
FN: klient miał churn, ale model go przeoczył
TN: klient nie miał churn i model przewidział brak churn
```


## Appendix C: przypomnienie parametrów modeli

### `DummyClassifier`

Baseline. W tym notebooku używamy:

```python
DummyClassifier(strategy="most_frequent")
```

Model zawsze wybiera najczęstszą klasę. Służy jako minimalny punkt odniesienia.

### `DecisionTreeClassifier`

Najważniejsze parametry:

- `max_depth` — maksymalna głębokość drzewa,
- `min_samples_leaf` — minimalna liczba obserwacji w liściu.

Większa głębokość zwiększa elastyczność, ale też ryzyko przeuczenia.

### `RandomForestClassifier`

Najważniejsze parametry:

- `n_estimators` — liczba drzew,
- `max_features` — ile cech losujemy przy szukaniu splitu,
- `min_samples_leaf` — minimalny rozmiar liścia,
- `n_jobs` — liczba rdzeni używana do obliczeń.

Random Forest zmniejsza wariancję pojedynczego drzewa przez uśrednianie wielu niezależnych drzew.

### `GradientBoostingClassifier`

Najważniejsze parametry:

- `n_estimators` — liczba kolejnych drzew-poprawek,
- `learning_rate` — wielkość kroku każdej poprawki,
- `max_depth` — głębokość pojedynczego małego drzewa.

Mały `learning_rate` zwykle wymaga większej liczby drzew.

### `XGBClassifier`

Najważniejsze parametry użyte w notebooku:

- `n_estimators` — liczba drzew,
- `learning_rate` — skala kolejnych poprawek,
- `max_depth` — maksymalna głębokość drzewa,
- `subsample` — część obserwacji używana do budowy drzewa,
- `colsample_bytree` — część cech używana do budowy drzewa,
- `reg_lambda` — regularyzacja wartości liści,
- `min_child_weight` — minimalna „waga” obserwacji w liściu,
- `eval_metric="logloss"` — metryka optymalizowana / raportowana podczas trenowania.

XGBoost jest nadal boostingiem drzew, ale z dodatkowymi mechanizmami kontroli złożoności i regularyzacji.


## Appendix D: `feature_importances_` — dokładniejsze przypomnienie

`feature_importances_` to liczby przypisane cechom po dopasowaniu modelu.
Wartości są zwykle znormalizowane tak, żeby ich suma wynosiła 1.

Interpretacja praktyczna:

```text
większa wartość -> cecha częściej / mocniej pomagała modelowi poprawiać predykcje
```

Dla drzew w scikit-learn jest to najczęściej tzw. **mean decrease impurity**:

```text
suma spadków nieczystości splitów używających danej cechy,
ważona liczbą obserwacji przechodzących przez te splity,
a potem znormalizowana.
```

Ograniczenia:

- ważność zależy od konkretnego modelu i zbioru danych,
- cechy skorelowane mogą dzielić między siebie ważność,
- cechy o wielu możliwych splitach mogą czasem wyglądać na ważniejsze,
- nie jest to analiza przyczynowa.

Do dokładniejszej analizy można użyć np. permutation importance albo SHAP, ale to byłby osobny temat.
